# 02 Dataset Preparation

## Overview

This notebook shows how to inspect, clean, split, and save a simple instruction dataset.
You will load the sample JSONL file from this repo and prepare it for training.

Estimated time: 30 to 40 minutes.

## Prerequisites

Complete notebooks 00 and 01 first.
You should know what an instruction response pair looks like.

In [ ]:
%pip install -q datasets==2.19.0 transformers==4.40.0

## Common Fine Tuning Formats

Fine tuning datasets often show up in one of these shapes:

* Alpaca format: `instruction`, `input`, `output`
* ShareGPT style chat turns
* Plain prompt and response pairs

The important point is not the file name.
The important point is that each example clearly teaches the behavior you want.

In [ ]:
from pathlib import Path
import sys
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

sys.path.append(str(Path("..").resolve()))

from utils.helpers import DATASETS_DIR, PREPARED_DATA_DIR

dataset_path = DATASETS_DIR / "sample_dataset.jsonl"
rows = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]
rows[:3]

## Quality Beats Quantity

Five hundred clean examples can beat fifty thousand noisy ones.
The model can only learn the pattern you show it.
If your dataset mixes styles, formats, or quality levels, the model learns that inconsistency too.

In [ ]:
raw_records = [
    {"question": "What is gradient descent?", "answer": "It is an optimization method that updates weights step by step to reduce loss."},
    {"question": "What is a token?", "answer": "It is a chunk of text the model turns into an ID before processing."},
]

converted_records = [
    {
        "instruction": item["question"],
        "input": "",
        "output": item["answer"],
    }
    for item in raw_records
]
converted_records

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

sample_prompt = (
    "### Instruction:\n" + rows[0]["instruction"] + "\n\n"
    "### Input:\n" + rows[0]["input"] + "\n\n"
    "### Response:\n" + rows[0]["output"]
)
token_ids = tokenizer(sample_prompt)["input_ids"]
print(f"Token count: {len(token_ids)}")
print(token_ids[:20])

In [ ]:
dataset = Dataset.from_list(rows)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
prepared = DatasetDict(train=split_dataset["train"], validation=split_dataset["test"])

print(prepared)

In [ ]:
output_dir = PREPARED_DATA_DIR
output_dir.mkdir(parents=True, exist_ok=True)
prepared["train"].to_json(output_dir / "train.jsonl")
prepared["validation"].to_json(output_dir / "validation.jsonl")
print(f"Saved prepared files to {output_dir.resolve()}")

## Key Takeaways

* Use a consistent schema before you train.
* Clean examples matter more than huge noisy dumps.
* Tokenization helps you estimate sequence length and cost.
* Always keep a validation split so you can check generalization.

## Next Steps

Continue to [03 LoRA Explained](../03-lora-explained/03-lora-explained.ipynb).